In [4]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from irrigation_env import IrrigationEnv, make_default_config

base_config = make_default_config(weather_seeds=(1,))

weather_file = Path(base_config["weather_file"])
weather_columns = tuple(base_config["weather_columns"])

raw_weather = pd.read_csv(
    weather_file,
    sep=r"\s+",
    header=None,
    names=weather_columns,
)

all_simyears = tuple(
    sorted(
        pd.to_numeric(raw_weather["simyear"])
        .astype(int)
        .unique()
        .tolist()
    )
)

print("Weather file:", weather_file)
print("Number of simyears:", len(all_simyears))
print("First simyears:", all_simyears[:10])
print("Last simyears:", all_simyears[-10:])

Weather file: CPWG.dat
Number of simyears: 1000
First simyears: (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
Last simyears: (991, 992, 993, 994, 995, 996, 997, 998, 999, 1000)


In [5]:
processing_config = make_default_config(
    weather_seeds=all_simyears
)

processing_env = IrrigationEnv(processing_config)

print("Processor:", processing_env.weather_processor)
print("Weather sequences:", len(processing_env._weather_bank))

/opt/anaconda3/envs/irrigation310/lib/python3.10/site-packages/aquacropgym/utils.py:18: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file,delim_whitespace=True,header=None)


Processor: aquacropgym.utils.calc_eto_faopm
Weather sequences: 1000


In [6]:
processed_frames = []

for simyear, weather_sequence in sorted(
    processing_env._weather_bank.items()
):
    number_of_days = len(weather_sequence.rain_mm)

    jdays = np.arange(
        processing_env.weather_start_jday,
        processing_env.weather_start_jday + number_of_days,
        dtype=np.int64,
    )

    processed_frames.append(
        pd.DataFrame(
            {
                "simyear": simyear,
                "jday": jdays,
                "rain_mm": weather_sequence.rain_mm,
                "eto_mm": weather_sequence.eto_mm,
            }
        )
    )

processed_weather = pd.concat(
    processed_frames,
    ignore_index=True,
)

print(processed_weather.head())
print(processed_weather.tail())
print("Shape:", processed_weather.shape)

   simyear  jday  rain_mm    eto_mm
0        1   121      0.0  5.998762
1        1   122      0.0  6.316588
2        1   123      0.0  6.174898
3        1   124      0.0  6.183888
4        1   125      0.0  6.581227
        simyear  jday  rain_mm    eto_mm
121995     1000   238      0.0  5.699756
121996     1000   239      0.0  6.831004
121997     1000   240      0.0  6.453374
121998     1000   241      0.0  5.133591
121999     1000   242      0.0  5.295542
Shape: (122000, 4)


In [ ]:
expected_days_per_simyear = (
    processing_env.episode_days
    + processing_env.forecast_horizon
    - 1
)

rows_per_simyear = processed_weather.groupby("simyear").size()

assert rows_per_simyear.eq(expected_days_per_simyear).all()
assert np.isfinite(
    processed_weather[["rain_mm", "eto_mm"]].to_numpy()
).all()
assert (processed_weather["rain_mm"] >= 0.0).all()
assert (processed_weather["eto_mm"] >= 0.0).all()

# ensure the year consistent with simyear
year_one = processed_weather.loc[
    processed_weather["simyear"] == 1
]

np.testing.assert_array_equal(
    year_one["rain_mm"].to_numpy(),
    processing_env._weather_bank[1].rain_mm,
)

np.testing.assert_array_equal(
    year_one["eto_mm"].to_numpy(),
    processing_env._weather_bank[1].eto_mm,
)

output_file = Path("CPWG_processed.csv")

processed_weather.to_csv(
    output_file,
    index=False,
    float_format="%.17g",
)

metadata = {
    "source_weather_file": str(weather_file),
    "processor": processing_env.weather_processor,
    "latitude_deg": processing_env.weather_latitude_deg,
    "altitude_m": processing_env.weather_altitude_m,
    "weather_start_jday": processing_env.weather_start_jday,
    "episode_days": processing_env.episode_days,
    "forecast_horizon": processing_env.forecast_horizon,
    "days_per_simyear": expected_days_per_simyear,
    "number_of_simyears": len(all_simyears),
    "columns": list(processed_weather.columns),
}

with Path("CPWG_processed_metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2)

print("Saved:", output_file.resolve())
print("Rows:", len(processed_weather))
print("Days per simyear:", expected_days_per_simyear)
print("ETo mean:", processed_weather["eto_mm"].mean())
print("ETo min:", processed_weather["eto_mm"].min())
print("ETo max:", processed_weather["eto_mm"].max())